# TFT Full Run — Kaggle GPU (Phase 2)
Full 100-epoch train of the Temporal Fusion Transformer lap-time model.

**Before running:**
1. Notebook settings (right panel): **Accelerator = GPU (P100 or T4)**, **Internet = ON**.
2. Add the data: **+ Add Input** → upload `tft_full_data.zip` as a new Dataset (or attach an existing one).
   The copy cell below auto-finds every `laps_*_r*.parquet` under `/kaggle/input/`, so the dataset name doesn't matter.

Bar to beat: LightGBM green val 2.18s (era0) / test 2.14s (era1).

In [ ]:
# 1. Install pf + lightning WITHOUT replacing Kaggle's GPU-matched torch.
# (Plain `pip install pytorch-forecasting` pulls a generic cu128 torch whose kernels
#  don't run on Kaggle's GPU -> "no kernel image". Pin torch to the preinstalled version
#  so the resolver leaves it untouched.)
# IMPORTANT: run this in a FRESH session (Run -> Restart & clear cell outputs, or restart
# the kernel) so torch is Kaggle's original build, not a clobbered one from a prior run.
import torch
KAGGLE_TORCH = torch.__version__.split('+')[0]
print('Preserving Kaggle torch:', torch.__version__)
!pip -q install "torch=={KAGGLE_TORCH}" "pytorch-forecasting==1.7.0" "lightning==2.6.5" mlflow fastf1 pandera scipy
import pytorch_forecasting as pf, lightning, torch
print('pf', pf.__version__, '| lightning', lightning.__version__, '| torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())

In [ ]:
# 2. Clone repo (or pull if already present) + add to path
import os, sys
REPO = '/kaggle/working/f1-strategist'
if not os.path.exists(REPO):
    !git clone https://github.com/Shreyansh262/f1-strategist.git $REPO
else:
    !cd $REPO && git pull
sys.path.insert(0, REPO)
os.chdir(REPO)
print('HEAD commit:')
!cd $REPO && git log --oneline -1

In [ ]:
# 3. Copy parquets from the attached Kaggle dataset into the repo's data/raw
import glob, shutil, pathlib
dst = pathlib.Path(REPO) / 'data' / 'raw'
dst.mkdir(parents=True, exist_ok=True)
src_files = glob.glob('/kaggle/input/**/laps_*_r*.parquet', recursive=True)
assert src_files, 'No laps_*_r*.parquet under /kaggle/input/ — attach the data zip as a Dataset first.'
for f in src_files:
    shutil.copy(f, dst / pathlib.Path(f).name)
copied = sorted(p.name for p in dst.glob('laps_*_r*.parquet'))
seasons = sorted({n.split('_')[1] for n in copied})
print(f'{len(copied)} parquet files in data/raw | seasons: {seasons}')

In [ ]:
# 4. Full run (100 epochs, EarlyStopping on val_loss). mlflow file-store opt-in is set in code.
from src.models.lap_time.train_tft import main
main(fast=False)

In [ ]:
# 5. Locate the CPU-loadable artifact to download (Output tab → right panel)
import glob
for p in glob.glob(f'{REPO}/models/*.pt') + glob.glob(f'{REPO}/models/*.ckpt'):
    print(p)